# Cattle Weight — Step A: Data Preparation
Run this once. Publish output as a Kaggle dataset (e.g. `cow-data-prep`). Steps B/C/D attach it.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "albumentations", "rembg[cpu]"], check=False)

In [ ]:
import os, json, re
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED   = 42
OUT    = Path('/kaggle/working')
INPUT  = Path('/kaggle/input')
IMG_EXTS = {'.jpg', '.jpeg'}
DATASET  = 'sadhliroomyprime/cattle-weight-detection-model-dataset-12k'

from functools import lru_cache
@lru_cache(maxsize=16)
def _count_imgs(p):
    try:
        count = 0
        for _, _, fnames in os.walk(str(p)):
            count += sum(1 for f in fnames if f.lower().endswith(('.jpg','.jpeg')))
        return count
    except Exception:
        return 0

EXPECTED = INPUT / 'cattle-weight-detection-model-dataset-12k'
if not EXPECTED.exists():
    candidates = sorted([(d, _count_imgs(d)) for d in INPUT.iterdir() if d.is_dir()],
                        key=lambda x: x[1], reverse=True)
    ROOT = candidates[0][0] if candidates else EXPECTED
    print(f'[WARN] auto-selected: {ROOT}')
else:
    ROOT = EXPECTED
print(f'ROOT: {ROOT}  ({_count_imgs(ROOT):,} images)')

def parse_filename_fields(stem):
    m = re.match(r'[\d.]+_s_(\d+)+_([\d.]+)_([MFmf])', stem)
    if m:
        w, a, s = float(m.group(1)), float(m.group(2)), m.group(3).upper()
        if 50 <= w <= 750: return w, a, s
    m2 = re.search(r'_(\d{2,4})_', stem)
    if m2:
        v = int(m2.group(1))
        if 50 <= v <= 750: return float(v), np.nan, ''
    return np.nan, np.nan, ''

def discover_dataset(root):
    file_idx, json_paths, csv_paths, img_paths = {}, [], [], []
    for f in root.rglob('*'):
        if not f.is_file(): continue
        ext = f.suffix.lower()
        file_idx[f.name] = f
        if ext == '.json': json_paths.append(f)
        elif ext == '.csv': csv_paths.append(f)
        elif ext in IMG_EXTS and '___fuse' not in f.name: img_paths.append(f)
    def resolve(fname):
        p = file_idx.get(Path(fname).name)
        return str(p) if p else ''
    records = []
    for jpath in json_paths:
        try:
            data = json.loads(jpath.read_text())
        except Exception: continue
        if not isinstance(data, dict) or 'images' not in data: continue
        ann_map = defaultdict(list)
        for a in data.get('annotations', []): ann_map[a['image_id']].append(a)
        for img in data['images']:
            fname = img.get('file_name', '')
            anns  = ann_map.get(img['id'], [])
            fn_w, fn_age, fn_sex = parse_filename_fields(Path(fname).stem)
            raw_w = img.get('weight', img.get('weight_kg',
                    anns[0].get('weight', np.nan) if anns else np.nan))
            try: w = float(raw_w)
            except Exception: w = fn_w
            if np.isnan(w): w = fn_w
            try: age = float(img.get('age', np.nan))
            except Exception: age = np.nan
            if np.isnan(age): age = fn_age
            sex = img.get('sex', img.get('gender', fn_sex)) or fn_sex
            bbox = None
            ann_bbox = [a for a in anns if 'bbox' in a and len(a['bbox']) == 4]
            if ann_bbox:
                bbox = max(ann_bbox, key=lambda a: a['bbox'][2]*a['bbox'][3])['bbox']
            records.append({'full_path': resolve(fname), 'file_name': fname,
                            'weight_kg': w, 'age': age, 'sex': sex, 'bbox': bbox,
                            'source': str(jpath.relative_to(root))})
    for cpath in csv_paths:
        try: dfc = pd.read_csv(cpath)
        except Exception: continue
        col_map = {}
        for c in dfc.columns:
            cl = c.lower()
            if 'weight' in cl: col_map[c] = 'weight_kg'
            elif 'age' in cl: col_map[c] = 'age'
            elif 'sex' in cl or 'gender' in cl: col_map[c] = 'sex'
            elif any(k in cl for k in ['file','image','img','name']): col_map[c] = 'file_name'
        dfc = dfc.rename(columns=col_map)
        if 'file_name' in dfc.columns and 'weight_kg' in dfc.columns:
            for _, row in dfc.iterrows():
                fname = str(row['file_name'])
                records.append({'full_path': resolve(fname), 'file_name': fname,
                                'weight_kg': float(row['weight_kg']) if pd.notna(row.get('weight_kg')) else np.nan,
                                'age': row.get('age', np.nan), 'sex': row.get('sex',''), 'bbox': None})
    covered = {r['file_name'] for r in records}
    for f in img_paths:
        if f.name not in covered:
            w, age, sex = parse_filename_fields(f.stem)
            if not np.isnan(w):
                records.append({'full_path': str(f), 'file_name': f.name,
                                'weight_kg': w, 'age': age, 'sex': sex, 'bbox': None})
    out = pd.DataFrame(records).drop_duplicates('file_name')
    return out[out['full_path'] != ''].reset_index(drop=True)

df_raw     = discover_dataset(ROOT)
df_labeled = df_raw.dropna(subset=['weight_kg']).copy()
print(f'Labeled: {len(df_labeled):,}  bbox_cov: {df_labeled["bbox"].notna().mean()*100:.1f}%')

WEIGHT_MIN, WEIGHT_MAX = 50, 700
df_labeled = df_labeled[df_labeled.weight_kg.between(WEIGHT_MIN, WEIGHT_MAX)].copy()
Q1, Q3 = df_labeled.weight_kg.quantile([0.25, 0.75])
df_labeled = df_labeled[df_labeled.weight_kg.between(Q1-4*(Q3-Q1), Q3+4*(Q3-Q1))].copy().reset_index(drop=True)
df_labeled['weight_bin'] = pd.qcut(df_labeled.weight_kg, q=10, labels=False, duplicates='drop')
train_df, temp_df = train_test_split(df_labeled, test_size=0.2, random_state=SEED, stratify=df_labeled.weight_bin)
val_df, test_df   = train_test_split(temp_df,    test_size=0.5, random_state=SEED, stratify=temp_df.weight_bin)
LOG_MEAN    = float(np.log(train_df.weight_kg).mean())
LOG_STD     = float(np.log(train_df.weight_kg).std())
WEIGHT_MEAN = float(train_df.weight_kg.mean())
WEIGHT_STD  = float(train_df.weight_kg.std())

# Serialize bbox as JSON string for CSV
for df in [train_df, val_df, test_df]:
    df['bbox'] = df['bbox'].apply(lambda x: json.dumps(x) if x is not None else 'null')

train_df.to_csv(OUT / 'train_split.csv', index=False)
val_df.to_csv(OUT   / 'val_split.csv',   index=False)
test_df.to_csv(OUT  / 'test_split.csv',  index=False)
json.dump({'log_mean': LOG_MEAN, 'log_std': LOG_STD,
           'weight_mean': WEIGHT_MEAN, 'weight_std': WEIGHT_STD,
           'n_train': len(train_df), 'n_val': len(val_df), 'n_test': len(test_df)},
          open(OUT / 'norm_state.json', 'w'), indent=2)
print(f'Saved splits: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')
print(f'norm_state.json: log_mean={LOG_MEAN:.4f}  log_std={LOG_STD:.4f}')


In [ ]:
import subprocess, sys, gc, os, concurrent.futures as cf
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rembg[cpu]'], check=False)
import cv2, numpy as np, tarfile
from pathlib import Path
from tqdm.auto import tqdm
from rembg import remove, new_session

OUT      = Path('/kaggle/working')
MASK_DIR = OUT / 'masks'
MASK_DIR.mkdir(exist_ok=True)
N_THREADS = max(1, min(4, os.cpu_count() or 2))
_GC_MAX_DIM = 512
_GC_ITERS   = 2

all_rows = pd.concat([train_df, val_df, test_df]).drop_duplicates('file_name').to_dict('records')
# Restore bbox (was serialized above)
import json
for r in all_rows:
    b = r.get('bbox')
    if isinstance(b, str) and b not in ('null',''):
        r['bbox'] = json.loads(b)
    elif b in (None, 'null', ''): r['bbox'] = None

missing = [r for r in all_rows
           if not (MASK_DIR / (Path(r['file_name']).stem + '.png')).exists()]
print(f'Total: {len(all_rows):,}  missing: {len(missing):,}')

def _has_bbox(r):
    b = r.get('bbox')
    return b is not None and isinstance(b, list)

def _grabcut_mask(row):
    img = cv2.imread(row['full_path'])
    if img is None: return False
    H0, W0 = img.shape[:2]
    bbox = row.get('bbox')
    if not isinstance(bbox, list): return False
    x, y, bw, bh = [max(0, int(v)) for v in bbox]
    scale = min(1.0, _GC_MAX_DIM / max(H0, W0, 1))
    if scale < 1.0:
        W_s = int(W0*scale); H_s = int(H0*scale)
        img_s = cv2.resize(img, (W_s, H_s), interpolation=cv2.INTER_AREA)
        x_s = int(x*scale); y_s = int(y*scale)
        bw_s = max(1,int(bw*scale)); bh_s = max(1,int(bh*scale))
    else:
        img_s = img; H_s,W_s = H0,W0; x_s,y_s,bw_s,bh_s = x,y,bw,bh
    pad = max(2, int(max(bw_s, bh_s)*0.04))
    rx = max(0, x_s-pad); ry = max(0, y_s-pad)
    rw = min(W_s-rx, bw_s+2*pad); rh = min(H_s-ry, bh_s+2*pad)
    if rw <= 0 or rh <= 0: return False
    try:
        mask_gc = np.zeros((H_s,W_s), np.uint8)
        bgd = np.zeros((1,65), np.float64); fgd = np.zeros((1,65), np.float64)
        cv2.grabCut(img_s, mask_gc, (rx,ry,rw,rh), bgd, fgd, _GC_ITERS, cv2.GC_INIT_WITH_RECT)
        alpha_s = np.where((mask_gc==cv2.GC_FGD)|(mask_gc==cv2.GC_PR_FGD), 255, 0).astype(np.uint8)
        alpha = cv2.resize(alpha_s, (W0,H0), interpolation=cv2.INTER_NEAREST) if scale<1.0 else alpha_s
        cv2.imwrite(str(MASK_DIR / (Path(row['file_name']).stem+'.png')), alpha)
        return True
    except Exception: return False

_SESS = None
def _rembg_mask(row):
    try:
        img = cv2.imread(row['full_path'])
        if img is None: return False
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        alpha = np.array(remove(rgb, session=_SESS))[:,:,3]
        cv2.imwrite(str(MASK_DIR / (Path(row['file_name']).stem+'.png')), alpha)
        return True
    except Exception: return False

if missing:
    gc_rows  = [r for r in missing if     _has_bbox(r)]
    rem_rows = [r for r in missing if not _has_bbox(r)]
    print(f'GrabCut: {len(gc_rows):,}  rembg: {len(rem_rows):,}')
    if gc_rows:
        with cf.ThreadPoolExecutor(max_workers=N_THREADS) as ex:
            gc_done = sum(tqdm(ex.map(_grabcut_mask, gc_rows), total=len(gc_rows), desc='GrabCut'))
        print(f'  GrabCut done: {gc_done:,}')
    if rem_rows:
        _SESS = new_session('u2netp')
        with cf.ThreadPoolExecutor(max_workers=N_THREADS) as ex:
            rem_done = sum(tqdm(ex.map(_rembg_mask, rem_rows), total=len(rem_rows), desc='rembg'))
        del _SESS; gc.collect()
        print(f'  rembg done: {rem_done:,}')

# Pack masks into a single tar.gz for easy sharing
print('Packing masks.tar.gz ...')
with tarfile.open(OUT / 'masks.tar.gz', 'w:gz') as t:
    t.add(MASK_DIR, arcname='masks')
print(f'masks.tar.gz: {(OUT/"masks.tar.gz").stat().st_size/1e6:.1f} MB')
print('\nOutputs ready to publish as a Kaggle dataset:')
for f in sorted(OUT.glob('*')):
    if f.is_file(): print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')
